# 01_play_history

DML: bronze_play_history — Raw listening history records.

In [ ]:
%run ../../tools/config/settings

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

raw_path = raw_base_path("play_history", ingestion_date, run_id)
raw_df   = spark.read.json(f"{raw_path}/*.json")

bronze = (
    raw_df
    .select(F.explode("items").alias("item"))
    .select(
        F.to_timestamp("item.played_at").alias("played_at"),
        F.col("item.track.id").alias("track_id"),
        F.col("item.track.name").alias("track_name"),
        F.col("item.track.duration_ms").alias("track_duration_ms"),
        F.col("item.track.artists.id").alias("artist_ids"),
        F.col("item.track.artists.name").alias("artist_names"),
        F.col("item.track.album.id").alias("album_id"),
        F.col("item.track.album.name").alias("album_name"),
        F.col("item.context.type").alias("context_type"),
        F.col("item.context.href").alias("context_href"),
        F.to_json(F.col("item")).alias("_raw"),
    )
    .withColumn("run_id",         F.lit(run_id))
    .withColumn("ingestion_date", F.to_date(F.lit(ingestion_date)))
)

bronze.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_play_history")
print(f"bronze_play_history: {bronze.count()} rows written")